# 🕒 Ejercicios InfluxDB

Este cuaderno está diseñado para explorar **el manejo del tiempo y las ventanas** utilizando SQL sobre InfluxDB. Aprenderás a trabajar con **series temporales**, ventanas fijas y móviles, y cómo agregar datos en diferentes intervalos.

> **Requisitos previos**:
> - Tener una instancia de **InfluxDB 3** (Cloud Serverless/Dedicated, Core o Enterprise).  
> - Contar con un **token** con permisos de **lectura y escritura**.  

## Pasos previos

In [0]:
# Instalar la librería de InfluxDB (pandas y matplotlib ya vienen instalados en Databricks)
%pip install influxdb3-python
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [influxdb3-python]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Ruta del volumen donde hemos subido los ficheros CSV de la carpeta data
data_path = "/Volumes/workspace/liga/datos/"

## 1) Conexión: `InfluxDBClient3`

In [0]:
# Configuración de la conexión a InfluxDB

# URL de la instancia de InfluxDB
HOST = "URL_DE_INFLUXDB"
# Organización (solo necesario en InfluxDB Cloud)
ORG = "NOMBRE_DE_LA_ORG"
# Base de datos (en InfluxDB Cloud se denomina bucket y es necesario crearlo antes)
DB = "NOMBRE_DEL_BUCKET/BASE_DE_DATOS"
# Token de acceso (guardado como secreto en Unity Catalog: workspace.liga.influxdb_token)
TOKEN = dbutils.secrets.get(catalog="workspace", schema="liga", key="influxdb_token")
# Si no se ejecuta en Databricks, comenta la línea anterior y escribe el token directamente:
# TOKEN = "TOKEN_DE_INFLUXDB"

In [0]:
from influxdb_client_3 import InfluxDBClient3

client = InfluxDBClient3(host=HOST, token=TOKEN, org=ORG, database=DB)

## 2) Ejercicios

### Carga de datos

### `football_events`

| Columna  | Tipo       | En InfluxDB | Descripción                                                          |
|----------|------------|-------------|----------------------------------------------------------------------|
| time     | TIMESTAMP  | timestamp   | Instante del evento (UTC)                                            |
| league   | STRING     | tag         | “LaLiga”, “Premier League”, “Serie A”, etc.                          |
| season   | STRING     | tag         | “2025/26”                                                            |
| match_id | STRING     | tag         | Identificador de partido (simulado)                                  |
| team     | STRING     | tag         | Equipo real (p. ej., “Real Madrid”)                                  |
| opponent | STRING     | tag         | Rival                                                                |
| player   | STRING     | tag         | Jugador real (p. ej., “Jude Bellingham”)                             |
| event    | STRING     | tag         | Tipo de evento: `pass`, `shot`, `goal`, `foul`, `save` o `xg_event`  |
| value    | DOUBLE     | field       | Valor del evento: 1 en `pass`, `goal`, `foul` y `save`; xG (entre 0 y 1) en `shot` y `xg_event` |
| minute   | INT        | field       | Minuto simulado (0–90)                                               |
| venue    | STRING     | tag         | “home” / “away”                                                      |

Las columnas de texto que usamos para **filtrar y agrupar** se cargan como **tags** (parámetro `tag_columns` de la celda siguiente) y las medidas numéricas como **fields**. En la salida de `SHOW COLUMNS` verás que las tags aparecen con el tipo `Dictionary(Int32, Utf8)`: InfluxDB las guarda como un diccionario de valores repetidos, lo que las hace muy eficientes para filtrar y agrupar.

In [0]:
client.write_file(file=data_path + "football_events.csv", tag_columns=["league", "season", "match_id", "team", "opponent", "player", "event", "venue"],data_format="csv")

query = """
SHOW COLUMNS IN football_events;
"""
table = client.query(query=query)
df = table.to_pandas()
display(df)

table_catalog,table_schema,table_name,column_name,data_type,is_nullable
public,iox,football_events,event,"Dictionary(Int32, Utf8)",YES
public,iox,football_events,league,"Dictionary(Int32, Utf8)",YES
public,iox,football_events,match_id,"Dictionary(Int32, Utf8)",YES
public,iox,football_events,minute,Int64,YES
public,iox,football_events,opponent,"Dictionary(Int32, Utf8)",YES
public,iox,football_events,player,"Dictionary(Int32, Utf8)",YES
public,iox,football_events,season,"Dictionary(Int32, Utf8)",YES
public,iox,football_events,team,"Dictionary(Int32, Utf8)",YES
public,iox,football_events,time,Timestamp(ns),NO
public,iox,football_events,value,Float64,YES


### Ejercicio 1

Usando la serie `football_events`, escribe una consulta que:

1. Considere solo los eventos de tipo **"goal"**.
2. Agrupe los resultados por `team` y cuente el número de goles de cada equipo.
3. Solo muestre los equipos que hayan marcado **2 goles o más**.
4. Ordene los resultados de mayor a menor cantidad de goles.
5. Solo muestre los **5 primeros** resultados.

🔍 **Pista:**
- Recuerda que `WHERE` filtra filas antes de la agregación, mientras que `HAVING` filtra después de agrupar.
- Para contar filas usa la función de agregación `COUNT(*)`, que devuelve el **número de filas** de cada grupo. Funciona igual que `AVG`, `MAX` o `MIN`, que ya viste en los ejemplos básicos:

    ```sql
    SELECT player, COUNT(*) AS num_registros
    FROM player_stats
    GROUP BY player
    ```

#### 🔹 La cláusula `HAVING` en SQL

La cláusula `HAVING` se utiliza para **filtrar resultados después de una agregación**. Esto es diferente de `WHERE`, que filtra **filas individuales antes de agrupar**.

##### 🔑 Diferencias entre `WHERE` y `HAVING`

| Cláusula | Cuándo se aplica | Ejemplo |
|----------|----------------|---------|
| `WHERE`  | Antes de la agregación, filtra filas | `WHERE event = 'goal'` |
| `HAVING` | Después de la agregación, filtra grupos | `HAVING COUNT(*) >= 2` |

In [0]:
query = """
SELECT
  team,
  COUNT(*) AS goals
FROM football_events
WHERE event = 'goal'
GROUP BY team
HAVING goals >= 2
ORDER BY goals DESC
LIMIT 5;
"""
table = client.query(query=query)
df = table.to_pandas()
display(df)

team,goals
VfB Stuttgart,55
Manchester City,44
AS Monaco,43
AC Milan,39
Lyon,39


### Ejercicio 2

Usando la serie `football_events`, escribe una consulta que:

1. Seleccione las columnas: `time`, `team`, `player`, `event` y `value`.
2. Filtre los registros para los **últimos 30 días** (hasta el momento actual).
3. Solo incluya eventos de la liga **"LaLiga"**.
4. Solo considere eventos del tipo **"shot"**.
5. Ordene los resultados por `time` de forma ascendente.
6. Solo muestre los **10 primeros** resultados.

🔍 **Pista:** utiliza `NOW()` y `INTERVAL` para calcular los últimos 30 días.

#### 💡 ¿Qué son las funciones `NOW()` e `INTERVAL` en SQL?

En SQL, la función `NOW()` devuelve la **fecha y hora actuales** del sistema en formato `TIMESTAMP`, lo que nos permite trabajar con el tiempo en tiempo real dentro de nuestras consultas.  

Para manipular fechas y horas, usamos la palabra clave `INTERVAL`, que permite **sumar o restar unidades de tiempo específicas**, como días, horas o minutos.  

**Ejemplos prácticos:**

```sql
-- Fecha y hora exacta hace 7 días
SELECT NOW() - INTERVAL '7 days';

-- Fecha y hora dentro de 2 horas
SELECT NOW() + INTERVAL '2 hours';
```

⚠️ **Ojo con el límite superior:** `time >= NOW() - INTERVAL '7 days'` incluye todo lo posterior a esa fecha, **también los eventos con fecha futura** si los hay (en este dataset los hay). Para quedarte de verdad con "los últimos 7 días" acota el rango por los dos extremos:

```sql
WHERE time >= NOW() - INTERVAL '7 days'
  AND time <= NOW()
```

📅 **Nota:** como `NOW()` cambia cada vez que ejecutas la consulta, los resultados de los ejercicios que la usan (del 2 al 6) **dependen del día y la hora** en que los ejecutes. Es normal que no coincidan exactamente con los de la solución.

In [0]:
query = """
SELECT time, team, player, event, value
FROM football_events
WHERE time >= now() - INTERVAL '30 days'
  AND time <= now()
  AND league = 'LaLiga'
  AND event = 'shot'
ORDER BY time ASC
LIMIT 10;
"""
table = client.query(query=query)
df = table.to_pandas()
display(df)

time,team,player,event,value
2026-11-05T16:47:19.000Z,Real Madrid,Vinicius Junior,shot,0.252
2026-11-05T16:47:27.000Z,Atletico de Madrid,Antoine Griezmann,shot,0.133
2026-11-05T16:47:58.000Z,Atletico de Madrid,Memphis Depay,shot,0.07
2026-11-05T16:49:18.000Z,Atletico de Madrid,Memphis Depay,shot,0.139
2026-11-05T16:51:26.000Z,Atletico de Madrid,Memphis Depay,shot,0.095
2026-11-05T16:52:50.000Z,Atletico de Madrid,Memphis Depay,shot,0.049
2026-11-05T16:55:12.000Z,Real Sociedad,Martin Zubimendi,shot,0.104
2026-11-05T16:56:23.000Z,Real Sociedad,Martin Zubimendi,shot,0.148
2026-11-05T16:57:27.000Z,Real Madrid,Vinicius Junior,shot,0.218
2026-11-05T16:57:40.000Z,Real Sociedad,Brais Mendez,shot,0.181


### Ejercicio 3

Usando la serie `football_events`, escribe una consulta que:

1. Cuente la cantidad de goles por equipo en los **últimos 5 días** (hasta el momento actual).
2. Solo considere eventos del tipo **"goal"**.
3. Agrupe los resultados por `team`.
4. Ordene los equipos de mayor a menor cantidad de goles.
5. Solo muestre los **5 primeros** resultados.

🔍 **Pista:** utiliza `NOW()` y `INTERVAL` para filtrar los últimos 5 días, y `COUNT(*)` para contar los goles.

In [0]:
query = """
SELECT team, COUNT(*) AS goals
FROM football_events
WHERE time >= NOW() - INTERVAL '5 days'
  AND time <= NOW()
  AND event = 'goal'
GROUP BY team
ORDER BY goals DESC
LIMIT 5;
"""
table = client.query(query=query)
df = table.to_pandas()
display(df)

team,goals
VfB Stuttgart,25
Bayer Leverkusen,23
Lille,20
Arsenal,19
Nice,14


### Ejercicio 4

Usando la serie `football_events`, escribe una consulta que:

1. Agrupe los eventos en ventanas de **15 minutos** usando `DATE_BIN`.  
2. Considere solo los eventos del **último día** (hasta el momento actual).  
3. Calcule el **número de tiros (shots)** por equipo en cada ventana.  
4. Agrupe los resultados por `team` y por ventana.  
5. Ordene los resultados por `team` y por ventana.
6. Solo muestre los **10 primeros** resultados.

🔍 **Pista:** son ventanas fijas. Lee la explicación de `DATE_BIN` y fíjate en qué devuelve: te servirá como columna de la ventana en el `GROUP BY` y en el `ORDER BY`.

#### 🕒 Ventanas Fijas (Tumbling Windows)

Muchas preguntas sobre series temporales son del tipo *"¿cuántos tiros hubo **en cada** cuarto de hora?"* o *"¿cuántos goles hubo **cada** hora?"*. Para responderlas se usan las **ventanas fijas** (tumbling windows): se **corta el tiempo en bloques consecutivos de la misma duración** y se calcula un agregado (conteo, suma, media…) para cada bloque.

Es como el resumen de un partido **por partes**: goles en la primera parte y goles en la segunda. Cada gol cuenta en una sola parte.

##### 📌 Ejemplo conceptual:

Tiros de un equipo y ventanas fijas de **15 minutos**:

| Tiro a las | Ventana a la que pertenece |
|------------|----------------------------|
| 10:02      | 10:00 – 10:15              |
| 10:09      | 10:00 – 10:15              |
| 10:14      | 10:00 – 10:15              |
| 10:21      | 10:15 – 10:30              |

Resultado:

| Ventana       | Tiros |
|---------------|-------|
| 10:00 – 10:15 | 3     |
| 10:15 – 10:30 | 1     |

- Las ventanas **no se solapan**: cada tiro pertenece a **una sola ventana**.
- Todas tienen la **misma duración** y van una detrás de otra, sin huecos.
- Se obtiene **una fila por ventana** (y por grupo, si agrupamos también por equipo): las filas originales se agrupan, igual que con un `GROUP BY` normal.

💡 **Tip:** son ideales para informes y métricas periódicas: goles por hora, visitas por día, transacciones por minuto, etc.

##### 💻 Ejemplo SQL de Ventanas Fijas

Supongamos que queremos **contar los goles por equipo cada 30 minutos**:

```sql
SELECT
    DATE_BIN(INTERVAL '30 minutes', time, TIMESTAMP '2025-01-01T00:00:00Z') AS window_start,
    team,
    COUNT(*) AS goals
FROM football_events
WHERE event = 'goal'
GROUP BY window_start, team
ORDER BY window_start, goals DESC;
```

🔍 Explicación del código:

- `DATE_BIN(INTERVAL '30 minutes', time, TIMESTAMP '2025-01-01T00:00:00Z')`: Asigna cada evento a su ventana de 30 minutos y devuelve el **inicio** de esa ventana. El tercer argumento es el **origen**: el instante a partir del cual se empiezan a cortar las ventanas (aquí, ventanas que empiezan en `:00` y `:30` de cada hora).
- `WHERE event = 'goal'`: Filtra por evento de tipo goal.
- `GROUP BY window_start, team`: Agrupa por ventana y equipo.
- `COUNT(*) AS goals`: Cuenta cuántos goles hubo en esa ventana.
- `ORDER BY window_start, goals DESC`: Ordena los resultados por ventana y cantidad de goles.

💡 Este patrón es muy útil para análisis de series temporales en Big Data.

In [0]:
query = """
SELECT
  DATE_BIN(INTERVAL '15 minutes', time, TIMESTAMP '2025-01-01T00:00:00Z') AS window_start,
  team,
  COUNT(*) AS shots
FROM football_events
WHERE event = 'shot'
  AND time >= now() - INTERVAL '1 day'
  AND time <= now()
GROUP BY team, window_start
ORDER BY team, window_start
LIMIT 10;
"""
table = client.query(query=query)
df = table.to_pandas()
display(df)

window_start,team,shots
2026-11-12T16:45:00.000Z,AS Roma,6
2026-11-12T17:00:00.000Z,AS Roma,5
2026-11-12T17:15:00.000Z,AS Roma,4
2026-11-12T17:30:00.000Z,AS Roma,4
2026-11-12T17:45:00.000Z,AS Roma,5
2026-11-12T18:00:00.000Z,AS Roma,5
2026-11-12T18:45:00.000Z,Athletic Club,2
2026-11-12T19:00:00.000Z,Athletic Club,4
2026-11-12T19:15:00.000Z,Athletic Club,3
2026-11-12T19:30:00.000Z,Athletic Club,5


### Ejercicio 5

Usando la serie `football_events`, escribe una consulta que:

1. Agrupe los eventos de tipo **"shot"** en ventanas de **30 minutos** usando `DATE_BIN`.  
2. Considere solo los eventos de los **últimos 5 días** (hasta el momento actual).  
3. Para cada equipo y ventana, calcule:  
   - `total_shots`: el **número total de tiros**.  
   - `last_shot_time`: la **hora del último tiro** dentro de la ventana.  
4. Agrupe los resultados por `team` y por ventana.  
5. Ordene los resultados por `team` y por ventana.
6. Solo muestre los **10 primeros** resultados.  

🔍 **Pista:**  
- Usa `COUNT(*)` para contar los tiros.  
- Para la hora del último tiro, usa una de las funciones `selector_*` que se explican a continuación.  


#### 🔹 Funciones `selector_*` en SQL

Las funciones `selector_*` son funciones de agregación propias de InfluxDB que permiten **seleccionar un valor concreto dentro de un grupo** (por ejemplo, el del último evento de cada ventana) y saber **en qué instante** se produjo. Son útiles cuando no te basta con agregados como `SUM`, `COUNT` o `MAX`, sino que quieres un dato de **una fila concreta** del grupo.

##### 🔑 Principales funciones `selector_*`

Todas reciben dos argumentos: `selector_xxx(valor, time)`, donde `valor` es la columna que quieres obtener y `time` es la columna de tiempo.

| Función                    | Qué fila selecciona dentro del grupo                       |
|----------------------------|------------------------------------------------------------|
| `selector_first(valor, time)` | La de `time` **más antiguo** (el primer evento).        |
| `selector_last(valor, time)`  | La de `time` **más reciente** (el último evento).       |
| `selector_min(valor, time)`   | La que tiene el **menor** `valor`.                      |
| `selector_max(valor, time)`   | La que tiene el **mayor** `valor`.                      |

##### 📦 ¿Qué devuelven?

No devuelven un número directamente, sino un **struct** (un pequeño registro) con dos campos de la fila seleccionada:

```
{ value: <valor de esa fila>, time: <time de esa fila> }
```

Para quedarte con uno de los dos campos se usan corchetes:

- `['value']` → el valor de la columna que has pedido.
- `['time']` → el instante en que ocurrió.

⚠️ Los campos se llaman siempre `value` y `time`, sea cual sea la columna. Por ejemplo, `selector_last(player, time)['player']` daría error; lo correcto es `['value']`.

##### 💻 Ejemplos prácticos

1. **Quién hizo el último tiro de cada ventana y cuándo:**

    ```sql
    selector_last(player, time)['value'] AS last_shooter,
    selector_last(player, time)['time']  AS last_shot_time
    ```

2. **Primer jugador que hizo un pase en cada ventana:**

    ```sql
    selector_first(player, time)['value'] AS first_passer
    ```

3. **Tiro con mayor xG (`value`) de cada ventana y cuándo se produjo:**

    ```sql
    selector_max(value, time)['value'] AS max_xg,
    selector_max(value, time)['time']  AS max_xg_time
    ```

##### 🔍 ¿Cómo funcionan?

- Se define un **grupo o ventana**, por ejemplo con `GROUP BY team, window_start`.
- Dentro de cada grupo, la función elige **una fila** según su criterio (primera, última, mínimo o máximo).
- Devuelve un struct con el `value` y el `time` de esa fila, del que extraes el campo que necesites con `['value']` o `['time']`.

💡 **Tip:** son muy útiles en análisis de **series temporales**, donde además de agregados como `SUM`, `COUNT` o `AVG` necesitas saber cuál fue el primer, último, mínimo o máximo valor de cada ventana de tiempo y cuándo ocurrió.

In [0]:
query = """
SELECT
  DATE_BIN(INTERVAL '30 minutes', time, TIMESTAMP '2025-01-01T00:00:00Z') AS window_start,
  team,
  COUNT(*) AS total_shots,
  selector_last(time, time)['time'] AS last_shot_time
FROM football_events
WHERE event = 'shot'
  AND time >= now() - INTERVAL '5 days'
  AND time <= now()
GROUP BY window_start, team
ORDER BY team, window_start
LIMIT 10;
"""
table = client.query(query=query)
df = table.to_pandas()
display(df)

window_start,team,total_shots,last_shot_time
2026-11-08T16:30:00.000Z,AC Milan,6,2026-11-08T16:56:34.000Z
2026-11-08T17:00:00.000Z,AC Milan,10,2026-11-08T17:25:28.000Z
2026-11-08T17:30:00.000Z,AC Milan,14,2026-11-08T17:59:19.000Z
2026-11-08T18:00:00.000Z,AC Milan,7,2026-11-08T18:14:33.000Z
2026-11-10T16:30:00.000Z,AS Monaco,3,2026-11-10T16:58:22.000Z
2026-11-10T17:00:00.000Z,AS Monaco,7,2026-11-10T17:26:43.000Z
2026-11-10T17:30:00.000Z,AS Monaco,7,2026-11-10T17:57:31.000Z
2026-11-10T18:00:00.000Z,AS Monaco,5,2026-11-10T18:17:05.000Z
2026-11-10T13:30:00.000Z,AS Roma,2,2026-11-10T13:54:23.000Z
2026-11-10T14:00:00.000Z,AS Roma,2,2026-11-10T14:25:33.000Z


### Ejercicio 6

Usando la serie `football_events`, escribe una consulta que:

1. Seleccione las columnas `time` y `player`.  
2. Calcule, para cada pase, el **número de pases que ha dado ese jugador en los 20 minutos anteriores** usando una ventana móvil (`COUNT(*) OVER (...)`).  
3. Filtre solo los eventos de tipo **"pass"** en los últimos 7 días (hasta el momento actual).  
4. Ordene los resultados por jugador y tiempo.  
5. Devuelva únicamente las primeras 10 filas.

🔍 **Pista:**  
- Usa `PARTITION BY player` para calcular la ventana individual por jugador.  
- Usa `RANGE INTERVAL '20 minutes' PRECEDING` para definir la ventana móvil de 20 minutos.

#### 🏃‍♂️ Ventanas Móviles (Sliding Windows)

Con las ventanas fijas respondemos a preguntas del tipo *"¿cuántos tiros hubo **en cada** cuarto de hora?"*. Pero a veces la pregunta es otra: *"¿cuántos tiros lleva el equipo **en los últimos** 30 minutos?"*. Para eso se usan las **ventanas móviles** (sliding windows).

En lugar de cortar el tiempo en bloques, **cada evento tiene su propia ventana**: la que termina en ese evento y mira hacia atrás una duración fija (por ejemplo, 30 minutos). Es como el marcador de *"tiros en los últimos 30 minutos"* que podría mostrar la televisión durante un partido: se actualiza con cada nuevo tiro.

##### 📌 Ejemplo conceptual:

Tiros de un equipo y ventana móvil de **15 minutos**:

| Tiro a las | Su ventana    | Tiros que entran      | Resultado |
|------------|---------------|-----------------------|-----------|
| 10:02      | 09:47 – 10:02 | 10:02                 | 1         |
| 10:09      | 09:54 – 10:09 | 10:02, 10:09          | 2         |
| 10:14      | 09:59 – 10:14 | 10:02, 10:09, 10:14   | 3         |
| 10:21      | 10:06 – 10:21 | 10:09, 10:14, 10:21   | 3         |

- El tiro de las **10:09** cuenta en **tres** ventanas distintas: las ventanas **se solapan**.
- A las 10:21 el tiro de las 10:02 ya ha **salido** de la ventana.
- Se obtiene **una fila por cada tiro**, cada una con su propio cálculo; no se agrupan las filas como con `GROUP BY`.

##### 🔑 Diferencias con las ventanas fijas

| Característica                        | Ventanas Fijas               | Ventanas Móviles             |
|---------------------------------------|------------------------------|------------------------------|
| **Duración**                          | Fija                         | Fija                         |
| **Solapamiento**                      | No                           | Sí                           |
| **Cada evento pertenece a**          | Una sola ventana             | Varias ventanas posibles     |
| **Filas de salida**                   | Una por ventana (y grupo)    | Una por evento               |
| **Ejemplo de uso**                    | Conteo de goles por hora     | Número de tiros en los 30 minutos anteriores a cada tiro |

##### 💻 Ejemplo SQL de Ventanas Móviles

Supongamos que queremos saber, **para cada tiro**, cuántos tiros ha hecho su equipo en los **30 minutos anteriores**:

```sql
SELECT
  time,
  team,
  COUNT(*) OVER (
    PARTITION BY team
    ORDER BY time
    RANGE INTERVAL '30 minutes' PRECEDING
  ) AS shots_last_30m
FROM football_events
WHERE event = 'shot'
  AND time >= NOW() - INTERVAL '7 days'
  AND time <= NOW()
ORDER BY team, time;
```

🔍 Explicación del código:

- `PARTITION BY team`: Separa los datos por equipo, para calcular la ventana móvil de forma independiente para cada equipo.
- `ORDER BY time`: Ordena los tiros por timestamp dentro de cada equipo, para que la ventana tenga sentido temporal.
- `RANGE INTERVAL '30 minutes' PRECEDING`: Define la ventana móvil: para cada tiro, incluye todos los tiros del equipo en los 30 minutos anteriores, incluido el propio tiro.
- `COUNT(*) OVER (...) AS shots_last_30m`: Cuenta cuántos tiros ha hecho el equipo en esa ventana de 30 minutos. Como las ventanas se solapan, un mismo tiro se cuenta en las ventanas de todos los tiros que lleguen en los 30 minutos siguientes.
- `WHERE event = 'shot' AND time >= NOW() - INTERVAL '7 days' AND time <= NOW()`: Filtra solo los tiros y limita el análisis a los últimos 7 días.
- `ORDER BY team, time`: Ordena los resultados por equipo y tiempo para que se vea claramente la evolución.

💡 **Tip:** Las ventanas móviles son perfectas para **métricas en tiempo real**, como el número de tiros en los últimos 30 minutos, el conteo de eventos recientes o para alertas de actividad inusual.

In [0]:
query = """
SELECT
  time,
  player,
  COUNT(*) OVER (
    PARTITION BY player
    ORDER BY time
    RANGE INTERVAL '20 minutes' PRECEDING
  ) AS passes_last_20m
FROM football_events
WHERE event = 'pass'
  AND time >= now() - INTERVAL '7 days'
  AND time <= now()
ORDER BY player, time
LIMIT 10;
"""
table = client.query(query=query)
df = table.to_pandas()
display(df)

time,player,passes_last_20m
2026-11-06T16:47:22.000Z,Achraf Hakimi,1
2026-11-06T16:51:44.000Z,Achraf Hakimi,2
2026-11-06T16:54:45.000Z,Achraf Hakimi,3
2026-11-06T17:00:12.000Z,Achraf Hakimi,4
2026-11-06T17:01:21.000Z,Achraf Hakimi,5
2026-11-06T17:02:28.000Z,Achraf Hakimi,6
2026-11-06T17:03:19.000Z,Achraf Hakimi,7
2026-11-06T17:21:08.000Z,Achraf Hakimi,4
2026-11-06T17:30:18.000Z,Achraf Hakimi,2
2026-11-06T17:44:05.000Z,Achraf Hakimi,2


## Cerrar conexión con InfluxDB

In [0]:
client.close()